<a href="https://colab.research.google.com/github/dokunoale/chagas/blob/feature%2FCNN/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!git clone --branch feature/CNN --single-branch https://github.com/dokunoale/chagas.git
%cd chagas
# Carica librerie installate
!pip install wfdb -q
!pip install gdown -q

# Aggiungi solo la root del progetto (src)
import sys
sys.path.append('/content/chagas/src')

# Importa tutto dai moduli
from preprocessing import tf_dataset_loader
from models import utils

# Importa simboli specifici (se vuoi)
from preprocessing.tf_dataset_loader import *
from models.utils import *

Cloning into 'chagas'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 119 (delta 17), reused 20 (delta 11), pack-reused 78 (from 1)
Receiving objects: 100% (119/119), 35.79 KiB | 939.00 KiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/chagas/chagas


In [10]:
import gdown

url = "https://drive.google.com/file/d/1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG/view?usp=drive_link"
gdown.download(url, "dataset.zip", quiet=False, fuzzy=True)
!unzip -q dataset.zip -d ./dataset

Downloading...
From (original): https://drive.google.com/uc?id=1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG
From (redirected): https://drive.google.com/uc?id=1RrxSAqML5xLWFSgilWLBv_X_H1vd04GG&confirm=t&uuid=10c422b8-dd2e-4bb6-9180-d4b5862b49a5
To: /content/chagas/chagas/dataset.zip
100%|██████████| 332M/332M [00:02<00:00, 112MB/s]


In [11]:
#Carichiamo su array np
X_train_positive, y_train_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/train/positives')
X_train_negative, y_train_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/train/negatives')

X_val_positive, y_val_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/val/positives')
X_val_negative, y_val_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/val/negatives')

X_test_positive, y_test_positive = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/test/positives')
X_test_negative, y_test_negative = tf_dataset_loader.load_dataset('/content/chagas/dataset/splitted_dataset/test/negatives')

Output streaming troncato alle ultime 5000 righe.
Parsing header for record: /content/chagas/dataset/splitted_dataset/val/positives/4249365
Fixing sample rate:
Loading record: /content/chagas/dataset/splitted_dataset/val/positives/872925
Parsing header for record: /content/chagas/dataset/splitted_dataset/val/positives/872925
Fixing sample rate:
Loading record: /content/chagas/dataset/splitted_dataset/val/positives/331311
Parsing header for record: /content/chagas/dataset/splitted_dataset/val/positives/331311
Fixing sample rate:
Loading record: /content/chagas/dataset/splitted_dataset/val/positives/350453
Parsing header for record: /content/chagas/dataset/splitted_dataset/val/positives/350453
Fixing sample rate:
Loading record: /content/chagas/dataset/splitted_dataset/val/positives/248547
Parsing header for record: /content/chagas/dataset/splitted_dataset/val/positives/248547
Fixing sample rate:
Loading record: /content/chagas/dataset/splitted_dataset/val/positives/438536
Parsing header

In [12]:
print(f"TRAIN: {X_train_positive.shape[0]} positivi, {X_train_negative.shape[0]} negativi")
print(f"VAL: {X_val_positive.shape[0]} positivi, {X_val_negative.shape[0]} negativi")
print(f"TEST: {X_test_positive.shape[0]} positivi, {X_test_negative.shape[0]} negativi")

TRAIN: 2090 positivi, 2110 negativi
VAL: 300 positivi, 300 negativi
TEST: 610 positivi, 590 negativi


In [21]:
print(f"TRAIN SHAPE: {X_train.shape} - VALIDATION SHAPE: {X_val.shape} - TEST SHAPE: {X_test.shape}")

TRAIN SHAPE: (4200, 2800, 12) - VALIDATION SHAPE: (600, 2800, 12) - TEST SHAPE: (1200, 2800, 12)


# MODELLO CNN

In [23]:
from models import layers

In [39]:
!git pull origin feature/CNN
%cd /content/chagas/src/models
from models import build_CNN
from build_CNN import build_ecg_model_with_spectrogram
import importlib
import build_CNN

importlib.reload(build_CNN)

from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import AUC
from sklearn.utils import class_weight

From https://github.com/dokunoale/chagas
 * branch            feature/CNN -> FETCH_HEAD
Already up to date.
/content/chagas/src/models


In [40]:
model = build_CNN.build_ecg_model_with_spectrogram ()

#compiliamo il modello
model.compile(optimizer='adam',
              loss=BinaryCrossentropy(),
              metrics=['accuracy', AUC(name='auc')])

#addestriamo il modello
callback = make_callback("1_CNN")

cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}

history1 = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=20,
                    batch_size=32,
                    class_weight=class_weights,
                    callbacks=callback)


Epoch 1/20
 34/132 ━━━━━━━━━━━━━━━━━━━━ 1:16 777ms/step - accuracy: 0.4840 - auc: 0.4449 - loss: 1.8879

KeyboardInterrupt: 